In [1]:
import os
os.environ['PYSPARK_PYTHON'] = r'C:\Tiago\data engineer\Projetos\credit-portfolio-analysis\venv\Scripts\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Tiago\data engineer\Projetos\credit-portfolio-analysis\venv\Scripts\python.exe'

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Credit Portfolio Analysis") \
    .master("local[*]") \
    .config("spark.pyspark.python", r"C:\Tiago\data engineer\Projetos\credit-portfolio-analysis\venv\Scripts\python.exe") \
    .config("spark.pyspark.driver.python", r"C:\Tiago\data engineer\Projetos\credit-portfolio-analysis\venv\Scripts\python.exe") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("SparkSession created successfully!")

Spark version: 3.5.1
SparkSession created successfully!


In [3]:
import sys
import os
sys.path.append(r'C:\Tiago\data engineer\Projetos\credit-portfolio-analysis\src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from data_generation import generate_portfolio
from transformations import apply_spark_transformations, apply_pandas_transformations
from analysis import pl_by_safra, default_rate_by_product, sensitivity_analysis
from visualizations import plot_net_margin_by_safra, plot_default_rate_by_product, plot_sensitivity_analysis

print("All libraries and modules imported successfully!")

All libraries and modules imported successfully!


In [4]:
# Generating synthetic credit portfolio data
data = generate_portfolio(n=10000, seed=42)
print(f"Total records generated: {len(data)}")

Total records generated: 10000


In [5]:
# Creating Spark DataFrame and applying transformations
df = spark.createDataFrame(data)
df = apply_spark_transformations(df)
df.show(5, truncate=False)

+---+---------+-----------------+--------------+-------------+--------------+-------------------------+------------------+-------+----------------+-------------+----------+
|age|client_id|employment_status|granted_amount|interest_rate|monthly_income|payment_status           |product           |safra  |interest_revenue|expected_loss|net_margin|
+---+---------+-----------------+--------------+-------------+--------------+-------------------------+------------------+-------+----------------+-------------+----------+
|19 |1        |Unemployed       |12999.7       |0.0241       |18926.44      |Adimplente               |Cartão de Crédito |2023-10|3759.51         |0.0          |3759.51   |
|55 |2        |Unemployed       |2460.06       |0.0292       |11415.16      |Parcialmente Inadimplente|Cartão de Crédito |2023-07|862.01          |984.02       |-122.01   |
|63 |3        |Self-Employed    |27702.13      |0.0293       |16772.28      |Parcialmente Inadimplente|Empréstimo Pessoal|2022-01|9740.

In [6]:
# Converting to Pandas and applying transformations
df_pandas = df.toPandas() if False else __import__('pandas').DataFrame(data)
df_pandas = apply_pandas_transformations(df_pandas)
print(df_pandas.head(5))

   client_id    safra             product  age employment_status  \
0          1  2023-10   Cartão de Crédito   19        Unemployed   
1          2  2023-07   Cartão de Crédito   55        Unemployed   
2          3  2022-01  Empréstimo Pessoal   63     Self-Employed   
3          4  2022-01  Empréstimo Pessoal   62          Employed   
4          5  2022-04       Financiamento   24        Unemployed   

   monthly_income  granted_amount  interest_rate             payment_status  \
0        18926.44        12999.70         0.0241                 Adimplente   
1        11415.16         2460.06         0.0292  Parcialmente Inadimplente   
2        16772.28        27702.13         0.0293  Parcialmente Inadimplente   
3        11431.44        14615.70         0.0290               Inadimplente   
4         9936.02        17853.83         0.0322                 Adimplente   

   interest_revenue  expected_loss  net_margin  
0           3759.51           0.00     3759.51  
1            862.0

In [7]:
# P&L by safra
pl_safra = pl_by_safra(df_pandas)
print(pl_safra.head(10))

     safra  total_clients  total_granted  total_revenue  total_loss  \
0  2022-01            438    11430956.72     6458084.55  4886812.27   
1  2022-02            343     8423468.01     4978965.36  2967006.72   
2  2022-03            424    10874374.49     6171401.61  4364223.35   
3  2022-04            401    10585629.69     5945539.38  4288465.91   
4  2022-05            439    10819836.67     6194140.11  4200300.91   
5  2022-06            424    11208868.10     6473823.40  4875727.13   
6  2022-07            417    10442142.56     6144495.72  4029829.06   
7  2022-08            411    10434570.97     6061041.17  4223869.16   
8  2022-09            390    10182275.45     5966990.65  3792926.40   
9  2022-10            433    11206270.57     6376833.67  4571827.62   

   total_net_margin  avg_interest_rate  
0        1571272.28               0.05  
1        2011958.64               0.05  
2        1807178.26               0.05  
3        1657073.47               0.05  
4        1993

In [8]:
# Default rate by product
default_df = default_rate_by_product(df_pandas)
print(default_df)

              product  default_rate
0   Cartão de Crédito         34.18
1          Consignado         33.88
2  Empréstimo Pessoal         32.39
3       Financiamento         32.02
